In [43]:
%pip install psycopg2 polars connectorx

   ---------------------------------------- 0.0/34.6 MB ? eta -:--:--
   -- ------------------------------------- 1.8/34.6 MB 11.9 MB/s eta 0:00:03
   ------ --------------------------------- 5.5/34.6 MB 16.0 MB/s eta 0:00:02
   ---------- ----------------------------- 9.4/34.6 MB 17.1 MB/s eta 0:00:02
   ------------- -------------------------- 12.1/34.6 MB 15.6 MB/s eta 0:00:02
   ------------------ --------------------- 15.7/34.6 MB 16.2 MB/s eta 0:00:02
   ---------------------- ----------------- 19.7/34.6 MB 16.7 MB/s eta 0:00:01
   --------------------------- ------------ 23.6/34.6 MB 17.0 MB/s eta 0:00:01
   ------------------------------- -------- 27.3/34.6 MB 17.2 MB/s eta 0:00:01
   ------------------------------------ --- 31.2/34.6 MB 17.4 MB/s eta 0:00:01
   ---------------------------------------  34.6/34.6 MB 17.5 MB/s eta 0:00:01
   ---------------------------------------- 34.6/34.6 MB 16.8 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.


In [45]:
import os
import pandas as pd
import polars as pl
from dotenv import load_dotenv, find_dotenv
import lightgbm as lgb
from sqlalchemy import create_engine
import psycopg2

In [55]:
def db_connection_string():
    load_dotenv(find_dotenv())

    host = 'localhost'
    port = 5433
    db = os.getenv('POSTGRES_DB')
    user = os.getenv('POSTGRES_USER')
    password = os.getenv('POSTGRES_PASSWORD')
    
    return f'postgresql://{user}:{password}@{host}:{port}/{db}'

In [57]:
def get_training_features():
    query = """
        SELECT *
        FROM dev_ml.all_features
    """
    conn_str = db_connection_string()
    
    df = pl.read_database_uri(query=query, uri=conn_str)

    decimal_cols = [c for c, dtype in df.schema.items() if dtype == pl.Decimal]
    df = df.with_columns([pl.col(c).cast(pl.Float64) for c in decimal_cols])
    df_pandas = df.to_pandas()

    return df_pandas

In [58]:
df_features = get_training_features()
df_features

,player_game_key,fixture_key,fpl_position,team_id,at_home,games_prior,appearance_rate,starting_rate,clean_sheet_rate,shot_conversion_rate,...,prev_season_influence_per_90,prev_season_creativity_per_90,prev_season_threat_per_90,prev_season_ict_index_per_90,prev_season_expected_goals_per_90,prev_season_expected_assists_per_90,prev_season_expected_goal_involvements_per_90,prev_season_expected_goal_chain_per_90,prev_season_expected_goal_buildup_per_90,league_avg_bps_this_season
0,2017_58_221,2017_58,DEF,43,False,5,0.6,0.666667,0.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
1,2017_58_222,2017_58,DEF,43,False,5,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
2,2017_58_223,2017_58,DEF,43,False,5,0.6,1.000000,0.333333,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
3,2017_58_224,2017_58,DEF,43,False,5,0.6,1.000000,0.333333,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
4,2017_58_225,2017_58,DEF,43,False,5,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254742,2017_57_99,2017_57,GKP,31,False,5,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
254743,2017_58_128,2017_58,DEF,43,False,5,1.0,0.800000,0.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
254744,2017_58_218,2017_58,GKP,43,False,5,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
254745,2017_58_219,2017_58,GKP,43,False,5,0.6,1.000000,0.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.823384
